# Dynamic Light Scattering, Zeta Potential, and Growth Inhibition Measurements for Self-Associating Amphiphiles Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.tnk3-xhn0/fair2.json`

### Notes
- All dataset entities should be referenced by their `@id` values whenever available.
- This notebook uses variables for dataset URLs, record set identifiers, and selected fields.
- The provided FAIR^2 package summary shows a rich dataset description, but the `recordSet` list in the package excerpt is empty. Because Croissant metadata may still resolve additional entities at runtime, this notebook first inspects the loaded metadata and then adapts to whatever record sets are available.


In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata from the dataset using `mlcroissant`, then inspect high-level dataset information.

Important: `dataset.metadata` is accessed as a single object, not treated like a dictionary or iterated over directly.

In [ ]:
import json
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 200)

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.tnk3-xhn0/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata_obj = dataset.metadata
metadata_json = metadata_obj.to_json()

print('Dataset loaded successfully.')
print('Name:', getattr(metadata_obj, 'name', 'N/A'))
print('Description:', getattr(metadata_obj, 'description', 'N/A'))
print('Version:', getattr(metadata_obj, 'version', 'N/A'))
print('License:', getattr(metadata_obj, 'license', 'N/A'))
print('URL:', croissant_url)

In [ ]:
# Optional: view a compact JSON preview of the metadata
metadata_preview = {
    'name': metadata_json.get('name'),
    'description': metadata_json.get('description'),
    'version': metadata_json.get('version'),
    'identifier': metadata_json.get('identifier'),
    'license': metadata_json.get('license'),
    'datePublished': metadata_json.get('datePublished')
}
print(json.dumps(metadata_preview, indent=2))

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

Because Croissant datasets can vary in structure, this section first discovers the record sets dynamically and then summarizes any fields found for each record set.

In [ ]:
def to_serializable_json(obj):
    """Convert Croissant objects to plain JSON-compatible Python structures when possible."""
    if hasattr(obj, 'to_json'):
        try:
            return obj.to_json()
        except Exception:
            pass
    if isinstance(obj, dict):
        return obj
    return None

record_sets_attr = getattr(metadata_obj, 'record_sets', None)
if record_sets_attr is None:
    record_sets_attr = getattr(metadata_obj, 'recordSet', None)

record_set_summaries = []

if record_sets_attr:
    for rs in record_sets_attr:
        rs_json = to_serializable_json(rs)
        if rs_json is None:
            rs_id = getattr(rs, 'id', None) or getattr(rs, '@id', None) or str(rs)
            rs_name = getattr(rs, 'name', None)
            fields = getattr(rs, 'fields', None) or getattr(rs, 'field', None) or []
            field_summaries = []
            for f in fields:
                f_json = to_serializable_json(f)
                if f_json is not None:
                    field_summaries.append({
                        '@id': f_json.get('@id') or f_json.get('id'),
                        'name': f_json.get('name'),
                        'dataType': f_json.get('dataType')
                    })
                else:
                    field_summaries.append({
                        '@id': getattr(f, 'id', None) or getattr(f, '@id', None) if hasattr(f, '@id') else str(f),
                        'name': getattr(f, 'name', None),
                        'dataType': getattr(f, 'data_type', None) if hasattr(f, 'data_type') else None
                    })
            record_set_summaries.append({
                '@id': rs_id,
                'name': rs_name,
                'field_count': len(field_summaries),
                'fields': field_summaries
            })
        else:
            fields = rs_json.get('field', []) or rs_json.get('fields', [])
            field_summaries = []
            for f in fields:
                if isinstance(f, dict):
                    field_summaries.append({
                        '@id': f.get('@id') or f.get('id'),
                        'name': f.get('name'),
                        'dataType': f.get('dataType')
                    })
                else:
                    field_summaries.append({'@id': str(f), 'name': None, 'dataType': None})
            record_set_summaries.append({
                '@id': rs_json.get('@id') or rs_json.get('id'),
                'name': rs_json.get('name'),
                'field_count': len(field_summaries),
                'fields': field_summaries
            })

overview_df = pd.DataFrame([
    {
        'record_set_@id': rs['@id'],
        'record_set_name': rs['name'],
        'field_count': rs['field_count']
    }
    for rs in record_set_summaries
])

if len(overview_df) == 0:
    print('No record sets were discovered in the loaded metadata.')
else:
    display(overview_df)

In [ ]:
# Show fields for each discovered record set using their @id values
if len(record_set_summaries) == 0:
    print('No record sets available to inspect.')
else:
    for rs in record_set_summaries:
        print('=' * 100)
        print('Record set @id:', rs['@id'])
        print('Record set name:', rs['name'])
        print('Field count:', rs['field_count'])
        field_df = pd.DataFrame(rs['fields'])
        if len(field_df) == 0:
            print('No fields listed for this record set.')
        else:
            display(field_df)

### Example record preview
The next cell prints a few records from the first available record set. This follows the template pattern while keeping the selection dynamic.

In [ ]:
if len(record_set_summaries) == 0:
    print('No record sets available, so no records can be previewed.')
else:
    preview_record_set_id = record_set_summaries[0]['@id']
    print('Previewing records from record set @id:', preview_record_set_id)
    try:
        for i, record in enumerate(dataset.records(record_set=preview_record_set_id)):
            print(record)
            if i >= 2:
                break
    except Exception as e:
        print('Could not iterate over records for this record set.')
        print(type(e).__name__, '-', e)

## 3. Data Extraction
Load data from each available record set into a pandas DataFrame for analysis.

Each DataFrame is stored in a dictionary keyed by the record set `@id`.

In [ ]:
record_set_ids = [rs['@id'] for rs in record_set_summaries if rs.get('@id')]
dataframes = {}
load_errors = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f'Loaded record set: {record_set_id} -> shape={dataframes[record_set_id].shape}')
    except Exception as e:
        load_errors[record_set_id] = f'{type(e).__name__}: {e}'
        print(f'Failed to load record set: {record_set_id} -> {type(e).__name__}: {e}')

print('\nLoaded DataFrames:', list(dataframes.keys()))
if load_errors:
    print('\nLoad errors:')
    for k, v in load_errors.items():
        print('-', k, '->', v)

In [ ]:
# Inspect the first successfully loaded record set
if len(dataframes) == 0:
    print('No DataFrames were loaded from the dataset.')
else:
    example_record_set_id = next(iter(dataframes))
    print('Example record set @id:', example_record_set_id)
    print('Columns:')
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())

### Dataset-wide summary of loaded tables
This gives a compact overview of the extracted tables.

In [ ]:
table_summary = pd.DataFrame([
    {
        'record_set_@id': rs_id,
        'n_rows': df.shape[0],
        'n_columns': df.shape[1]
    }
    for rs_id, df in dataframes.items()
]).sort_values(['n_rows', 'n_columns'], ascending=False) if len(dataframes) > 0 else pd.DataFrame()

if len(table_summary) == 0:
    print('No loaded tables to summarize.')
else:
    display(table_summary)

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps to one selected record set.

This section is written to adapt automatically:
- it chooses a record set that contains at least one numeric column,
- selects one numeric field for filtering and normalization,
- optionally groups by a categorical field if one exists.

All choices are displayed explicitly so you can replace them with specific record set and field `@id` values after inspection.

In [ ]:
def find_numeric_columns(df):
    numeric_cols = []
    for col in df.columns:
        converted = pd.to_numeric(df[col], errors='coerce')
        non_null = converted.notna().sum()
        if non_null > 0:
            numeric_cols.append(col)
    return numeric_cols

def find_categorical_columns(df):
    categorical_cols = []
    for col in df.columns:
        series = df[col]
        if series.dtype == 'object' or str(series.dtype).startswith('string'):
            if series.nunique(dropna=True) > 0:
                categorical_cols.append(col)
    return categorical_cols

selected_record_set_id = None
selected_numeric_field = None
selected_group_field = None

for rs_id, df in dataframes.items():
    numeric_candidates = find_numeric_columns(df)
    if len(numeric_candidates) > 0:
        selected_record_set_id = rs_id
        selected_numeric_field = numeric_candidates[0]
        categorical_candidates = find_categorical_columns(df)
        selected_group_field = categorical_candidates[0] if len(categorical_candidates) > 0 else None
        break

print('Selected record set @id:', selected_record_set_id)
print('Selected numeric field:', selected_numeric_field)
print('Selected group field:', selected_group_field)

In [ ]:
if selected_record_set_id is None or selected_numeric_field is None:
    print('No suitable numeric field was found for EDA.')
else:
    eda_df = dataframes[selected_record_set_id].copy()
    numeric_field = selected_numeric_field
    group_field = selected_group_field

    eda_df[numeric_field] = pd.to_numeric(eda_df[numeric_field], errors='coerce')
    eda_df = eda_df.dropna(subset=[numeric_field]).copy()

    print('Working DataFrame shape after numeric conversion:', eda_df.shape)
    print('Numeric summary:')
    display(eda_df[[numeric_field]].describe())

    threshold = eda_df[numeric_field].median()
    filtered_df = eda_df[eda_df[numeric_field] > threshold].copy()

    print(f'Filtered records with {numeric_field} > median threshold ({threshold}):')
    display(filtered_df.head())

    std_value = filtered_df[numeric_field].std()
    if pd.notna(std_value) and std_value != 0:
        filtered_df[f'{numeric_field}_normalized'] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / std_value
        )
    else:
        filtered_df[f'{numeric_field}_normalized'] = 0.0

    print(f'Normalized {numeric_field} for filtered records:')
    display(filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())

    if group_field is not None and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field, dropna=False)[numeric_field].agg(['count', 'mean', 'median', 'std']).reset_index()
        print(f'Grouped data by {group_field}:')
        display(grouped_df.head(20))
    else:
        print('No suitable grouping field was available in the selected record set.')

In [ ]:
# Optional outlier handling using the IQR rule
if selected_record_set_id is None or selected_numeric_field is None:
    print('Skipping outlier analysis because no numeric field is available.')
else:
    outlier_df = dataframes[selected_record_set_id].copy()
    outlier_df[selected_numeric_field] = pd.to_numeric(outlier_df[selected_numeric_field], errors='coerce')
    outlier_df = outlier_df.dropna(subset=[selected_numeric_field]).copy()

    q1 = outlier_df[selected_numeric_field].quantile(0.25)
    q3 = outlier_df[selected_numeric_field].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    no_outliers_df = outlier_df[
        (outlier_df[selected_numeric_field] >= lower_bound) &
        (outlier_df[selected_numeric_field] <= upper_bound)
    ].copy()

    print('IQR-based outlier handling summary')
    print('Field:', selected_numeric_field)
    print('Lower bound:', lower_bound)
    print('Upper bound:', upper_bound)
    print('Original rows:', len(outlier_df))
    print('Rows without outliers:', len(no_outliers_df))

## 5. Visualization
Visualize the selected numeric field and, when possible, show grouped summaries.

The plotting code is defensive so that it still runs even when only a small amount of data is available.

In [ ]:
if selected_record_set_id is None or selected_numeric_field is None:
    print('No suitable data available for visualization.')
else:
    viz_df = dataframes[selected_record_set_id].copy()
    viz_df[selected_numeric_field] = pd.to_numeric(viz_df[selected_numeric_field], errors='coerce')
    viz_df = viz_df.dropna(subset=[selected_numeric_field]).copy()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.histplot(viz_df[selected_numeric_field], kde=True, ax=axes[0], color='steelblue')
    axes[0].set_title(f'Distribution of {selected_numeric_field}')
    axes[0].set_xlabel(selected_numeric_field)

    sns.boxplot(y=viz_df[selected_numeric_field], ax=axes[1], color='lightcoral')
    axes[1].set_title(f'Boxplot of {selected_numeric_field}')
    axes[1].set_ylabel(selected_numeric_field)

    plt.tight_layout()
    plt.show()

In [ ]:
if selected_record_set_id is None or selected_numeric_field is None:
    print('Skipping grouped visualization because no numeric field is available.')
else:
    plot_df = dataframes[selected_record_set_id].copy()
    plot_df[selected_numeric_field] = pd.to_numeric(plot_df[selected_numeric_field], errors='coerce')
    plot_df = plot_df.dropna(subset=[selected_numeric_field]).copy()

    if selected_group_field is not None and selected_group_field in plot_df.columns:
        top_groups = (
            plot_df[selected_group_field]
            .astype(str)
            .value_counts(dropna=False)
            .head(10)
            .index
        )
        grouped_plot_df = plot_df[plot_df[selected_group_field].astype(str).isin(top_groups)].copy()

        plt.figure(figsize=(12, 6))
        sns.boxplot(data=grouped_plot_df, x=selected_group_field, y=selected_numeric_field)
        plt.xticks(rotation=45, ha='right')
        plt.title(f'{selected_numeric_field} by {selected_group_field}')
        plt.tight_layout()
        plt.show()
    else:
        print('No suitable categorical grouping field was available for grouped visualization.')

## 6. Conclusion
This notebook demonstrated how to:

- load a Croissant dataset from a URL with `mlcroissant`,
- inspect dataset metadata without treating `dataset.metadata` as a dictionary-like object,
- discover record sets and fields using their `@id` values,
- extract records from available record sets into pandas DataFrames,
- perform adaptive exploratory data analysis on a numeric field,
- visualize distributions and grouped summaries when the necessary columns are present.

### Next steps
- Replace the automatically selected record set and field names with specific `@id` values once you identify the tables most relevant to your analysis.
- Add domain-specific analyses for DLS, zeta potential, or growth inhibition data after inspecting which record sets are exposed by the Croissant metadata.
- Extend the notebook with joins across record sets if common identifiers are present.
